# Figure 6 and Table 15

Noun purity, synonym top-10 accuracy and split-half cosine of the ten finetuned models, before finetuning and after finetuning on SPEECH-COCO with written or with spoken captions, and the checkpoint kept for each run.

Inputs: `results/finetune_effect.parquet` and `results/finetune_draws.parquet` from `scripts/analysis/figure6_finetune.py`; the `provenance.json` of each checkpoint from `scripts/finetune/train.py`; `corpus/speech-coco_provenance.json` and `corpus/speech-coco_manifest.parquet` from `scripts/finetune/speech_coco.py`; `lexicon/words.parquet` from `scripts/stimuli/wordlists.py`.

In [ ]:
import json
import math
from decimal import ROUND_HALF_UP, Decimal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import font_manager as fm
from matplotlib.colors import to_rgb
from matplotlib.textpath import TextPath

from src import neighbors
from src.config import CHECKPOINTS, CORPUS
from src.finetune.recipe import schedule
from src.models import (DEFAULT_SWEEP, FAMILY_OF, FINETUNES, MODELS, PARAMS_M, PRETTY,
                        TWIN_OF, finetune_key)
from src.utils import setup_notebook, viz
from src.utils.results import read, words

setup_notebook()
viz.use_real_bold()

EFFECT = read("finetune_effect")
DRAWS = read("finetune_draws")
RECORDS = pd.json_normalize([
    json.loads((CHECKPOINTS / key / "provenance.json").read_text()) for key in FINETUNES])
CORPUS_RECORD = json.loads((CORPUS / "speech-coco_provenance.json").read_text())
CAPTIONS = pd.read_parquet(CORPUS / "speech-coco_manifest.parquet",
                           columns=["image_id", "speaker", "speed"])

#: The frozen encoder of each run: written captions through a text encoder,
#: spoken captions through a speech encoder.
WRITTEN, SPEECH = "qwen3-0p6b-frozen", "w2v-bert-2.0-frozen"

#: Word images per word: the four backgrounds of one typography setting, and all 96.
IMAGES, ALL_IMAGES = 4, 96

#: The ten pretrained models that are finetuned.
BACKBONES = tuple(dict.fromkeys(TWIN_OF[key] for key in FINETUNES))


def rounded(value: float, places: int = 2) -> str:
    """``value`` at ``places`` decimals, rounded half up as the paper prints it."""
    return str(Decimal(repr(round(float(value), 10))).quantize(
        Decimal(1).scaleb(-places), rounding=ROUND_HALF_UP))

## Table 15

The step of the checkpoint kept for each run, with an asterisk for an early stop.

In [ ]:
RUN = {WRITTEN: "written", SPEECH: "speech"}
KEPT = RECORDS.assign(
    Model=RECORDS.base_model.map(PRETTY),
    run=RECORDS.text_tower.map(RUN),
    step=[f"{step:,}" + ("*" if outcome.startswith("early stop") else "")
          for step, outcome in zip(RECORDS.train_step, RECORDS.outcome)])
TABLE15 = (KEPT.pivot(index="Model", columns="run", values="step")
           .loc[KEPT.Model.unique(), list(RUN.values())].rename_axis(columns=None))
display(TABLE15)
display(RECORDS.assign(model=RECORDS.model.map(PRETTY))
        [["model", "train_step", "best_step", "outcome"]])

per_model = RECORDS.groupby("base_model").text_tower.agg(["size", "nunique"])
not_finetuned = [PRETTY[m] for m in DEFAULT_SWEEP
                 if MODELS[m].kind in ("dino", "mae") and m not in BACKBONES]
print(f"finetuned models: {len(per_model)}; self-supervised models not finetuned: "
      f"{', '.join(not_finetuned)}; runs per model: "
      f"{', '.join(map(str, sorted(set(per_model['size']))))}, against "
      f"{', '.join(map(str, sorted(set(per_model['nunique']))))} different encoders")
print(f"runs that keep the step with the best validation loss: "
      f"{(RECORDS.train_step == RECORDS.best_step).sum()} of {len(RECORDS)}")

stopped = RECORDS.outcome.str.extract(r"early stop at step (\d+)")[0].dropna().astype(int)
plan = schedule(int(RECORDS.corpus_kept.iloc[0]))
last_validation = plan["steps"] // plan["validate_every"] * plan["validate_every"]
print(f"runs stopped early: {len(stopped)} of {len(RECORDS)}")
print(f"runs stopped early at step {last_validation:,}, the last validation "
      f"of the schedule: {(stopped == last_validation).sum()}")

## Schedule and corpus

The twenty runs share one schedule, set by the size of the corpus after the text filter.

In [ ]:
def recorded(column: str) -> str:
    """The distinct values of one provenance field over the twenty runs."""
    return ", ".join(f"{v:,}" for v in sorted(set(RECORDS[column])))


print(f"epochs: {recorded('max_epochs')}; steps: {recorded('total_steps')}")
print(f"warmup: {recorded('recipe.warmup')} steps; validation every "
      f"{plan['validate_every']} steps")
print(f"batch: {recorded('recipe.batch')} images per step; projection width: "
      f"{recorded('recipe.width')}")
print(f"early stop after {recorded('patience')} validations without improvement")
print(f"captions after the text filter: {recorded('corpus_kept')}")
print(f"spoken captions per image: {len(CAPTIONS) / CAPTIONS.image_id.nunique():.1f}")
rates = sorted(CAPTIONS.speed.unique())
print(f"voices: {CAPTIONS.speaker.nunique()}; speaking rates: {len(rates)} "
      f"({', '.join(f'{v:.1f}' for v in rates)})")
print(f"images held out for validation: {CORPUS_RECORD['val_percent']} percent")

## The five draws

The typography setting of each draw of four word images, the same before and after finetuning.

In [ ]:
FOUR = (DRAWS[DRAWS.images == IMAGES].drop(columns=["images", "manifest_sha"])
        .set_index("draw"))
display(FOUR)
counts = FOUR.appearance.value_counts()
print(f"draws: {len(FOUR)}; distinct typography settings: {FOUR.appearance.nunique()}; "
      f"drawn twice: setting {', '.join(map(str, counts[counts == 2].index))}")

## Values cited in the text

Scores from four word images are means over the five draws.

In [ ]:
K = 10
INDEXED = EFFECT.set_index(["model", "readout", "task", "k", "images"])


def value(model: str, readout: str, task: str, k: int, images: int = IMAGES) -> float:
    """One readout of one model, the mean over the five draws when it uses four word images."""
    return float(INDEXED.value[(model, readout, task, k, images)])


QUOTED = {"noun purity": ("purity", "common_noun", K),
          "synonym top-10 accuracy": ("retrieval", "synonym", K),
          "capital top-10 accuracy": ("retrieval", "country_capital", K)}
CHANGE = pd.DataFrame([
    dict(model=model, readout=name, images=images, run=RUN[tower],
         before=value(model, *key, images),
         after=value(finetune_key(model, tower), *key, images))
    for model in BACKBONES for name, key in QUOTED.items()
    for images in (IMAGES, ALL_IMAGES) for tower in RUN
]).assign(change=lambda f: f.after - f.before)

print("DINOv3-H, before finetuning -> after the written run, after the speech run")
for (name, images), rows in CHANGE[CHANGE.model == "dinov3-h"].groupby(
        ["readout", "images"], sort=False):
    print(f"  {name} with {images} images: {rounded(rows.before.iloc[0])} -> "
          + ", ".join(rounded(v) for v in rows.after))

written_h = CHANGE[(CHANGE.model == "dinov3-h") & (CHANGE.run == "written")].set_index(
    ["readout", "images"])
print("DINOv3-H after the written run with 4 images, against before finetuning with all 96: "
      + "; ".join(f"{name} {rounded(written_h.after[(name, IMAGES)])} against "
                  f"{rounded(written_h.before[(name, ALL_IMAGES)])}"
                  for name in ("noun purity", "synonym top-10 accuracy")))
capital_spread = INDEXED.spread[("dinov3-h", *QUOTED["capital top-10 accuracy"], IMAGES)]
print(f"DINOv3-H capital top-10 accuracy with 4 images: standard deviation over the "
      f"{len(FOUR)} draws before finetuning {rounded(capital_spread)}, change after the "
      f"written run {rounded(written_h.change[('capital top-10 accuracy', IMAGES)])}")

print("largest absolute change after either run, with 4 or all 96 word images:")
for name in QUOTED:
    changes = CHANGE[CHANGE.readout == name]
    h = changes[changes.model == "dinov3-h"].change.abs().max()
    rest = changes[changes.model != "dinov3-h"]
    top = rest.loc[rest.change.abs().idxmax()]
    print(f"  {name}: DINOv3-H {h:.3f}; the other nine models at most "
          f"{abs(top.change):.3f} ({PRETTY[top.model]})")

CHANCE_PURITY = neighbors.chance_purity(
    words().query("role == 'category'").category.to_numpy())
print(f"chance noun purity, the dashed line in panel (a): {rounded(CHANCE_PURITY)}")

## Rows, lanes and readouts

Each model is a row with one lane per run, grouped by family and drawn in color when Section 4 finds semantic structure in it.

In [ ]:
#: The two runs as ``(tower, fill of the finetuned mark, name in the key)``.
#: "solid" fills the mark with the row's color, "tint" with that color laid
#: over white.
RUNS = ((WRITTEN, "solid", "written"), (SPEECH, "tint", "spoken"))

#: The frozen encoder each run is trained against, the key's second line.
TOWER = {WRITTEN: "Qwen3-Embedding-0.6B", SPEECH: "w2v-BERT 2.0"}

#: The models Section 4 finds semantic structure in, drawn in color.
COLOURED = ("dinov3-h", "dinov3-l")

FAMILY_ORDER = ("DINOv3", "DINOv2", "MAE", "Web-DINO")

#: The models of each family, largest first.
BANDS = tuple(
    (family, tuple(sorted((m for m in BACKBONES if FAMILY_OF[m] == family),
                          key=lambda m: -PARAMS_M[m])))
    for family in FAMILY_ORDER)

#: ``(readout, task, k, panel name)``.
READOUTS = (
    ("purity", "common_noun", K, "Noun purity"),
    ("retrieval", "synonym", K, "Synonym"),
    ("reliability", "tau", 0, "Signal strength"),
)

SUBTITLE = {"purity": f"{K} nearest nouns",
            "retrieval": f"Top-{K} accuracy",
            "reliability": "Split-half cosine similarity"}

## Geometry

Sizes are in points, and each panel's limits hold every drawn value with the room its mark needs.

In [ ]:
#: Ink geometry of a mark, in points.
MARK = viz.MARK_SIZE
RAD = (MARK / np.pi) ** 0.5
RING_LW = 0.7
TINT_LW = 0.5
CONNECTOR_LW = 0.8

#: The least white left between the two marks of a lane before the open mark is
#: dropped and the line alone carries the value before finetuning.
WHITE_GAP_PT = 0.8

#: Center-to-center distance at which the open mark clears the filled one.
RING_CLEAR_PT = RAD + RING_LW / 2 + RAD + TINT_LW / 2 + WHITE_GAP_PT

HEIGHT_PT = 214.0
GUTTER_PT = 48.0
PANEL_PT = 100.0

#: Where each run's lane sits, in row units from the row's center.
LANE = dict(zip((tower for tower, _, _ in RUNS), (0.22, -0.22)))

#: Clearance each kind of mark needs inside the panel, in points.
MARK_ROOM_PT = RAD + 0.9
TICK_ROOM_PT = 4.0
CHANCE_ROOM_PT = 7.0

#: Tick steps on a 1-2-2.5-5 ladder, largest first. Four ticks are preferred to three.
TICK_STEPS = sorted({s * 10.0 ** e for e in range(-4, 1) for s in (1.0, 2.0, 2.5, 5.0)},
                    reverse=True)
TICK_COUNTS = (4, 3)

#: Header baselines in points above the panel's top spine.
HEADER_BASE_Y = 1.0
HEADER_NAME_RISE_PT = 11.5
HEADER_SUBTITLE_RISE_PT = 3.5

#: Space between the ink of the panel letter and the panel name.
HEADER_LETTER_GAP_PT = 3.2

#: Bands reserved above the panels for the header and below them for the key.
HEADER_BAND_PT = 23.0
LEGEND_BAND_PT = 21.0


def row_plan() -> tuple[list[tuple], list[tuple]]:
    """``([(y, model, colour)], [(family, y_top, y_bottom)])``, top to bottom."""
    plan, spans, y = [], [], 0.0
    for family, models in BANDS:
        top = y + 0.5
        for m in models:
            plan.append((y, m, viz.SEMANTIC if m in COLOURED else viz.CONTROL))
            y -= 1.0
        spans.append((family, top, y + 0.5))
    return plan, spans


ROWS, SPANS = row_plan()
ROW_MODELS = [m for _, m, _ in ROWS]
YLIM = (min(y for y, *_ in ROWS) - 0.5, 0.5)
SHADED = set(FAMILY_ORDER[::2])


def band_edges(top, bottom):
    """Band limits in data units, tiled so adjacent bands meet exactly."""
    return (YLIM[1] if top >= YLIM[1] else top,
            YLIM[0] if bottom <= YLIM[0] else bottom)


def panel_anchors(readout, task, k):
    """Every value the panel draws, with the clearance in points each one needs."""
    anchors = [(value(m, readout, task, k), MARK_ROOM_PT) for m in ROW_MODELS]
    anchors += [(value(finetune_key(m, tower), readout, task, k), MARK_ROOM_PT)
                for m in BACKBONES for tower, _, _ in RUNS]
    anchors += [(value(m, readout, task, k, ALL_IMAGES), TICK_ROOM_PT)
                for m in ROW_MODELS]
    if readout == "purity":
        anchors.append((CHANCE_PURITY, CHANCE_ROOM_PT))
    return anchors


def tick_range(lo, hi, step):
    """The multiples of ``step`` between ``lo`` and ``hi``."""
    return tuple(round(i * step, 10)
                 for i in range(math.ceil(lo / step), math.floor(hi / step) + 1))


def fitted_limits(anchors, panel_pt=PANEL_PT):
    """Limits that hold every anchor with its own clearance, and their ticks.

    The clearance is in points and the limits in data units, so the span is
    iterated to its fixed point.
    """
    span = max(v for v, _ in anchors) - min(v for v, _ in anchors)
    lo = hi = 0.0
    for _ in range(64):
        lo = min(v - room * span / panel_pt for v, room in anchors)
        hi = max(v + room * span / panel_pt for v, room in anchors)
        if abs((hi - lo) - span) < 1e-12:
            break
        span = hi - lo
    ticks = next(t for wanted in TICK_COUNTS for step in TICK_STEPS
                 for t in [tick_range(lo, hi, step)] if len(t) == wanted)
    return (lo, hi), ticks


LIMITS, TICKS = zip(*(fitted_limits(panel_anchors(*r[:3])) for r in READOUTS))

## Drawing and key

The open mark is left out where it would not clear the filled mark, and the line alone shows the change.

In [ ]:
def tint(colour, weight=0.58):
    """``colour`` laid over the white ground at ``weight``."""
    rgb = np.array(to_rgb(colour))
    return tuple(1.0 - weight * (1.0 - rgb))


def tick_label(tick: float) -> str:
    """A tick as the paper prints a proportion, without its leading zero."""
    text = f"{tick:.3f}".rstrip("0").rstrip(".")
    return text if text in ("0", "1") else text.removeprefix("0")


def ink_width(text: str, size: float, weight: str = "normal") -> float:
    """The right edge of ``text``'s ink at ``size``, from its drawing origin."""
    prop = fm.FontProperties(family=plt.rcParams["font.sans-serif"][0], weight=weight)
    return float(TextPath((0, 0), text, size=size, prop=prop).get_extents().x1)


def cap_height(size: float) -> float:
    """The height of a capital letter at ``size`` points, in points."""
    prop = fm.FontProperties(family=plt.rcParams["font.sans-serif"][0])
    return float(TextPath((0, 0), "H", size=size, prop=prop).get_extents().height)


#: One dotted leader along each row's center.
LEADER = dict(color=viz.GRID, lw=viz.LEADER_WIDTH, ls=viz.DASH["construction"], zorder=0)

#: Chance noun purity, at the weight of the bottom spine.
CHANCE_RULE = dict(color=viz.CHANCE, ls=viz.DASH["chance"], lw=0.45, zorder=1)

#: The value with all 96 word images: one upright through both lanes of a row.
REFERENCE_TICK_HALF = max(abs(v) for v in LANE.values()) + 0.21
REFERENCE_TICK = dict(color=viz.RULE, lw=0.45, solid_capstyle="butt", zorder=1.5)


def panel_scale(fig, ax) -> float:
    """Points of panel per unit of the quantity, once the layout has settled."""
    box = ax.get_window_extent(fig.canvas.get_renderer())
    lo, hi = ax.get_xlim()
    return (box.width * 72.0 / fig.dpi) / (hi - lo)


def draw_lane(ax, before, after, lane, colour, fill, scale) -> None:
    """One run: the line, the open mark before finetuning, the filled mark after.

    The open mark is drawn only when it clears the filled one; otherwise the
    line alone trails from under the filled mark on the side the model came from.
    """
    separation = abs(after - before) * scale
    if separation > 0.0:
        ax.plot([before, after], [lane, lane], color=colour, lw=CONNECTOR_LW,
                solid_capstyle="butt", zorder=2)
    if separation >= RING_CLEAR_PT:
        ax.scatter(before, lane, s=MARK, marker="o", facecolor=viz.WHITE,
                   edgecolor=colour, linewidth=RING_LW, zorder=3)
    ax.scatter(after, lane, s=MARK, marker="o",
               facecolor=colour if fill == "solid" else tint(colour),
               edgecolor=colour if fill == "tint" else "none",
               linewidth=TINT_LW if fill == "tint" else 0.0, zorder=4)


def draw_rows(ax, readout, task, k, scale) -> None:
    """Every row of one panel."""
    if readout == "purity":
        ax.axvline(CHANCE_PURITY, **CHANCE_RULE)
    for y, model, colour in ROWS:
        ax.plot([0, 1], [y, y], transform=ax.get_yaxis_transform(), **LEADER)
        full = value(model, readout, task, k, ALL_IMAGES)
        ax.plot([full, full], [y - REFERENCE_TICK_HALF, y + REFERENCE_TICK_HALF],
                **REFERENCE_TICK)
        before = value(model, readout, task, k)
        for tower, fill, _ in RUNS:
            after = value(finetune_key(model, tower), readout, task, k)
            draw_lane(ax, before, after, y + LANE[tower], colour, fill, scale)


def draw_gutter(gax):
    """The left column, which carries the model names."""
    gax.set_axis_off()
    gax.set_xlim(0, 1)
    gax.set_ylim(*YLIM)
    for y, model, _ in ROWS:
        gax.text(1.0, y, PRETTY[model], transform=gax.get_yaxis_transform(),
                 ha="right", va="center", fontsize=viz.TEXT_SIZE, color=viz.CONTROL)


def draw_bands(axes):
    """Shade every other family inside each panel, behind everything."""
    for ax in axes[1:]:
        for family, top, bottom in SPANS:
            if family in SHADED:
                hi, lo = band_edges(top, bottom)
                ax.axhspan(lo, hi, facecolor=viz.FAINT, edgecolor="none", zorder=-1)


def draw_header(ax, letter, name, subtitle):
    """The panel letter and name, over the quantity plotted."""
    indent = ink_width(letter, viz.PANEL_SIZE, "bold") + HEADER_LETTER_GAP_PT
    for dx, rise, text, size, colour, weight in (
            (0.0, HEADER_NAME_RISE_PT, letter, viz.PANEL_SIZE, viz.INK, "bold"),
            (indent, HEADER_NAME_RISE_PT, name, viz.PANEL_SIZE, viz.INK, "bold"),
            (0.0, HEADER_SUBTITLE_RISE_PT, subtitle, viz.SUBTITLE_SIZE, viz.CONTROL,
             "normal")):
        t = ax.annotate(text, (0.0, HEADER_BASE_Y), xycoords="axes fraction",
                        xytext=(dx, rise), textcoords="offset points",
                        ha="left", va="baseline", fontsize=size, color=colour,
                        fontweight=weight, annotation_clip=False)
        # Kept out of the layout solve, so the header does not narrow the panel.
        t.set_in_layout(False)


def style_panel(ax, limits, ticks):
    ax.set_xlim(*limits)
    ax.set_ylim(*YLIM)
    ax.set_yticks([])
    ax.set_xticks(list(ticks), [tick_label(t) for t in ticks])
    ax.tick_params(axis="x", labelsize=viz.TEXT_SIZE, length=2, pad=1.5)
    ax.spines["left"].set_visible(False)
    ax.spines["bottom"].set_linewidth(0.45)


KEY_LINE_GAP_PT = 8.5
KEY_GLYPH_GAP_PT = 4.5
KEY_TOP_PT = 4.0
KEY_RIGHT_SLACK_PT = 3.0
#: The key's marks, a little larger than a plot mark, and the length of its line.
KEY_MARK = MARK * 1.25
KEY_DUMBBELL_PT = 15.0


def key_items() -> list[tuple[str, str, str]]:
    """``[(kind, name, second line)]``, left to right."""
    return [(fill, f"Before and after, {name}", TOWER[tower]) for tower, fill, name in RUNS] + [
        ("reference", f"All {ALL_IMAGES} word images", "before finetuning")]


def draw_key_glyph(kax, kind, left, y, colour):
    """One key glyph, drawn at ``left`` on the baseline ``y``."""
    if kind == "reference":
        kax.plot([left + 0.5] * 2, [y - 3.4, y + 3.4], **REFERENCE_TICK)
        return
    right = left + KEY_DUMBBELL_PT
    kax.plot([left, right], [y, y], color=colour, lw=CONNECTOR_LW,
             solid_capstyle="butt", zorder=2)
    kax.scatter(left, y, s=KEY_MARK, marker="o", facecolor=viz.WHITE,
                edgecolor=colour, linewidth=RING_LW, zorder=3, clip_on=False)
    kax.scatter(right, y, s=KEY_MARK, marker="o",
                facecolor=colour if kind == "solid" else tint(colour),
                edgecolor=colour if kind == "tint" else "none",
                linewidth=TINT_LW if kind == "tint" else 0.0, zorder=4, clip_on=False)


def draw_key(fig, axes):
    """One line of items under the panels, spread from the first panel to the last."""
    fig.canvas.draw()
    width_pt = fig.get_figwidth() * 72.0
    kax = fig.add_axes((0.0, 0.0, 1.0, LEGEND_BAND_PT / HEIGHT_PT))
    kax.set_in_layout(False)
    kax.set_axis_off()
    kax.set_xlim(0.0, width_pt)
    kax.set_ylim(0.0, LEGEND_BAND_PT)
    title_y = LEGEND_BAND_PT - KEY_TOP_PT - cap_height(viz.TITLE_SIZE)
    second_y = title_y - KEY_LINE_GAP_PT
    glyph_y = (LEGEND_BAND_PT - KEY_TOP_PT + second_y) / 2.0

    renderer = fig.canvas.get_renderer()
    placed = []
    for kind, name, second in key_items():
        texts = [kax.text(0.0, title_y, name, fontsize=viz.TITLE_SIZE,
                          color=viz.INK, ha="left", va="baseline"),
                 kax.text(0.0, second_y, second, fontsize=viz.TEXT_SIZE,
                          color=viz.CONTROL, ha="left", va="baseline")]
        text_w = max(t.get_window_extent(renderer).width for t in texts) * 72.0 / fig.dpi
        glyph_w = 1.0 if kind == "reference" else KEY_DUMBBELL_PT
        placed.append((kind, texts, glyph_w, glyph_w + KEY_GLYPH_GAP_PT + text_w))

    left = axes[1].get_position().x0 * width_pt
    right = axes[-1].get_position().x1 * width_pt - KEY_RIGHT_SLACK_PT
    gap = (right - left - sum(w for *_, w in placed)) / (len(placed) - 1)
    x = left
    for kind, texts, glyph_w, w in placed:
        draw_key_glyph(kax, kind, x, glyph_y, viz.CONTROL)
        for t in texts:
            t.set_x(x + glyph_w + KEY_GLYPH_GAP_PT)
        x += w + gap


def draw_figure():
    fig, axes = plt.subplots(
        1, 1 + len(READOUTS), figsize=viz.inches(HEIGHT_PT, viz.TEXT_WIDTH_IN),
        layout="constrained",
        gridspec_kw=dict(width_ratios=(GUTTER_PT,) + (PANEL_PT,) * len(READOUTS)))
    fig.get_layout_engine().set(
        w_pad=0.012, h_pad=0.010, wspace=0.045,
        rect=(0, LEGEND_BAND_PT / HEIGHT_PT, 1,
              1 - (HEADER_BAND_PT + LEGEND_BAND_PT) / HEIGHT_PT))
    draw_gutter(axes[0])
    for i, (ax, readout, limits, ticks) in enumerate(zip(axes[1:], READOUTS, LIMITS, TICKS)):
        draw_header(ax, f"({chr(97 + i)})", readout[3], SUBTITLE[readout[0]])
        style_panel(ax, limits, ticks)
    # Whether a lane keeps its open mark depends on how far apart two values
    # land in points, which is known once the layout has run.
    fig.canvas.draw()
    scales = [panel_scale(fig, ax) for ax in axes[1:]]
    for ax, r, scale in zip(axes[1:], READOUTS, scales):
        draw_rows(ax, *r[:3], scale)
    draw_bands(axes)
    draw_key(fig, axes)
    return fig

In [ ]:
fig = draw_figure()
viz.save(fig, "figure6_finetune")
plt.show()